[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adaptionlabs/adaption-runbooks/blob/main/adaptive-data/invent_a_dataset.ipynb)

# Invent a Dataset

Generate a dataset from scratch for training language models using free-form prompting. No source dataset required to get started.

In this Colab, we will use a detailed `dataset_prompt` to generate a training-ready dataset of multi-turn medical conversations.

**Docs:** [/adaptive-data/invent-a-dataset/](https://docs.adaptionlabs.ai/adaptive-data/invent-a-dataset/) · [runbook page](https://docs.adaptionlabs.ai/runbooks/invent-a-dataset/)  
**Blog:** [Invent a Dataset: Data Generation for Custom Models](https://adaptionlabs.ai/blog/introducing-invent-a-dataset) · [Measuring Dataset Generation Abilities with Zero Seed Data](https://adaptionlabs.ai/blog/measuring-dataset-generation-abilities-with-zero-seed-data)

## 1. Install `adaption`

In [ ]:
!pip install adaption

## 2. Instantiate the Adaption Python API client

If you don't have an Adaption API key then refer [this](https://docs.adaptionlabs.ai/introduction/getting-started/#create-an-api-key)  documentation to quickly create one.

In [ ]:
import time
from adaption import Adaption
from google.colab import userdata


client = Adaption(api_key=userdata.get('ADAPTION_API_KEY'))

## 3. Estimate the invention

We will start with a request to estimate the cost of inventing a 100 row dataset. This will allow us to see the credit cost before triggering the full run.



In [ ]:
estimate = client.datasets.invent(
    dataset_prompt=(
        "A dataset containing multi-turn conversations between a seasoned doctor and a "
        "medical student covering diagnoses, treatments, drug explanations, and "
        "healthcare roles. Prompts must be structured as a JSON array of messages "
        "containing 'role' and 'content' keys, while responses should be plain text."
    ),
    rows=100,
    estimate=True,
)
print(f"Estimated credits: {estimate.estimated_credits} | Available Credits: {estimate.available_credits}",)

## 4. Invent!

Now that we have priced out the run, we can remove `estimate=True` and create a name for our invented dataset.

Below, we also specify that we would like the output as an `instruction_dataset`. If you would like to invent preference pairs for alignment, pass `training_type="preference_pairs"`.

### Translation & Localization

Invent also supports expanding the dataset to other languages and locales. Check out the [documentation](https://docs.adaptionlabs.ai/adaptive-data/invent-a-dataset/#expand-across-languages-and-locales) to learn more!

In [ ]:
dataset = client.datasets.invent(
    name="Clinical Conversations",
    dataset_prompt=(
        "A dataset containing multi-turn conversations between a seasoned doctor and a "
        "medical student covering diagnoses, treatments, drug explanations, and "
        "healthcare roles. Prompts must be structured as a JSON array of messages "
        "containing 'role' and 'content' keys, while responses should be plain text."
    ),
    rows=100,
    training_type="instruction_dataset",
)
dataset_id = dataset.id
print(dataset_id, dataset.status)  # running

## 5. Wait for invention to complete

We run the invention process asynchonously on our platform. Below we use the `wait_for_completion` helper to poll the status of the dataset until it's ready to be used.

In [ ]:
record = client.datasets.wait_for_completion(dataset_id)
print(f"Generated {record.row_count} rows")


## 6. Download the dataset

After the dataset is ready, we can download it to a local `jsonl` file for inspection and usage in our own products later on.

In [ ]:
client.datasets.download(
    dataset_id,
    file_format="jsonl",
).write_to_file("clinical-qa.jsonl")

## 7. View the datatset

In [ ]:
import pandas as pd
from google.colab import files
from google.colab.data_table import DataTable

df = pd.read_json('clinical-qa.jsonl', lines=True)
DataTable(df, include_index=False)